In [1]:
import pandas as pd
import tarfile
import xml.etree.ElementTree as ET
from xml.dom.minidom import parse, parseString
from collections import Counter
import numpy as np
import re
import csv
from tqdm import tqdm
import csv
import pickle

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
#open the tar.gz file as a tar archive
tar_archive = tarfile.open(name=r"C:\Users\bbinnend\Desktop\D&E\Thesis\Data\2021-01.tar.gz", mode="r:gz")

#extract the tar archive in a directory of choice
#tar_archive.extractall("data/2021-01-01.2021-31-01/")

#Initialize an empty list and fill it with path to each .xml file
xml_files_list=[]

for file in tar_archive.getnames():
    if file[-4:] == ".xml":
        xml_files_list.append(str("Data/2021-01-01.2021-31-01/" + file))

tar_archive.close()

--------------------------------------------------------------------------- <br>
PRE-SELECTING FILES BASED ON PRESENCE OF SECTIONS <br>
--------------------------------------------------------------------------- <br>

The general idea is so only select documents which have a contract section written in english, containing all of the required sections

In [5]:
def get_xmlns_value(file):
    """This function takes in an xml file and returns the namespace of the file"""
    root_tag = str(root.tag)
    xmlns_value = root_tag.split("}")[0] + "}"
    xmlns_len = len(xmlns_value)
    return xmlns_value

In [6]:
#this cell contains code which filters all the document based on the presence of a set of elements
required_sections = [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY",
                     ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT"
                     ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE",
                     ".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT",
                     ]

changing the query in the required sections to include a formsection in english reduces the amount of data significantly from 543 to 35 <br>
how does filtering on english bias the data?

In [13]:
#print all attributes, values and texts of x elements in data_dict
pre_filtered_docs = []

for i, file in tqdm(enumerate(xml_files_list[0:10000])):
    #parse the tree and get the root element
    tree = ET.parse(file)
    root = tree.getroot()
    xmlns_value = get_xmlns_value(file)
    
    for section in required_sections:
        query = section.format(xmlns_value, xmlns_value, xmlns_value, xmlns_value)
        if len(root.findall(query)) > 0:
            status = "accepted"
        else:
            status = "reject"
    
    if status == "accepted":
        pre_filtered_docs.append(file)
    else:
        continue

10000it [01:23, 119.99it/s]


In [15]:
#The code below is used to save the created dictionary
with open("prefiltered_document_list.pickle", "wb") as f:
    pickle.dump(pre_filtered_docs, f)